In [11]:
# =======================================
# 0. Import & cấu hình cơ bản
# =======================================
import os
import re
import glob
from typing import Dict, List, Optional

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from pymongo import MongoClient
from datetime import datetime

# Đường dẫn thư mục chứa tất cả CSV điểm (mỗi file = 1 môn)
DATA_DIR = "output_final_fixed"                     # TODO: sửa lại cho đúng thư mục của bạn
WEIGHTS_FILE = "./weight/weights_used.csv"  # TODO: sửa lại nếu cần
OUTPUT_DIR = "./output_final_v2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =======================================
# 1. Định nghĩa schema chuẩn & alias
# =======================================

# Cột target
TARGET_COL = "final"

# Các feature điểm (KHÔNG gồm final)
FEATURE_SCORE_COLS = [
    "attend",
    "quiz",
    "quiz2",
    "midterm",
    "homework",
    "homework1",
    "homework2",
    "group_project",
    "individual_project",
    "practice",
    "regular",
    "speech_and_discussion",
    "project"
]

# Toàn bộ feature điểm + target (dùng cho tạo mask, xử lý chung)
CANONICAL_FEATURES = FEATURE_SCORE_COLS + [TARGET_COL]

# Cột ID/meta
ID_COLUMNS = ["student_id", "course_code", "no"]

# Map alias để nhận diện cột raw -> cột chuẩn
COLUMN_ALIASES: Dict[str, List[str]] = {
    "no": ["no", "No"],
    "student_id": ["student id", "student_id", "id"],
    "course_code": ["course_code", "course code", "course"],

    "attend": ["attend", "attendance"],
    "quiz": ["quiz"],
    "quiz2": ["quiz2"],
    "homework": ["homework", "hw", "assignment"],
    "homework1": ["homework1", "hw1"],
    "homework2": ["homework2", "hw2"],
    "midterm": ["midterm", "mid term"],
    "group_project": ["group project", "group_project"],
    "individual_project": ["individual project", "individual_project"],
    "practice": ["practice", "lab", "exercise"],
    "regular": ["regular", "participation", "classwork"],
    "speech_and_discussion": ["speech and discussion", "speech_and_discussion"],
    "project": ["project"],
    "final": ["final", "final_exam"]
}

def normalize_col_name(col: str) -> str:
    """Chuẩn hóa tên cột: lower, strip, thay space bằng '_'."""
    return re.sub(r"\s+", "_", str(col).strip().lower())

def find_canonical_for_raw(raw_col: str) -> Optional[str]:
    """Tìm xem cột raw (sau normalize) map được vào canonical nào."""
    raw_norm = normalize_col_name(raw_col)
    for canonical, aliases in COLUMN_ALIASES.items():
        for alias in aliases:
            alias_norm = normalize_col_name(alias)
            if raw_norm == alias_norm:
                return canonical
    return None

# =======================================
# 2. Load weights_used.csv & tạo baseline
# =======================================

def load_weights(weights_file: str) -> Dict[str, Dict[str, float]]:
    """
    Đọc weights_used.csv và trả về:
    {
      course_code: {
          canonical_feature: weight,
          ...
      },
      ...
    }
    """
    if not os.path.exists(weights_file):
        print("Không tìm thấy weights_used.csv, bỏ qua phần baseline.")
        return {}

    wdf = pd.read_csv(weights_file)
    # Chuẩn hóa course_code & component
    if "course_code" not in wdf.columns or "component" not in wdf.columns or "weight" not in wdf.columns:
        print("File weights_used.csv không đúng format kỳ vọng (cần course_code, component, weight).")
        return {}

    wdf["course_code"] = wdf["course_code"].astype(str).str.strip()
    wdf["component_norm"] = wdf["component"].apply(find_canonical_for_raw)

    # Bỏ các component không map được
    wdf = wdf[~wdf["component_norm"].isna()].copy()

    weights_by_course: Dict[str, Dict[str, float]] = {}
    for _, row in wdf.iterrows():
        course = row["course_code"]
        comp = row["component_norm"]
        weight = float(row["weight"])
        if course not in weights_by_course:
            weights_by_course[course] = {}
        weights_by_course[course][comp] = weights_by_course[course].get(comp, 0.0) + weight

    print("Đã load weights cho", len(weights_by_course), "course.")
    return weights_by_course

def add_baseline_from_weights(data: pd.DataFrame,
                              weights_by_course: Dict[str, Dict[str, float]]) -> pd.DataFrame:
    """
    Tính baseline_final_weighted = sum(component_score * weight)
    cho từng course_code (nếu có weight). Bỏ qua component = final
    để tránh dùng target trực tiếp.
    """
    data = data.copy()
    data["baseline_final_weighted"] = np.nan

    if not weights_by_course:
        print("Không có weights_by_course, baseline_final_weighted sẽ toàn NaN.")
        return data

    # Chuẩn hóa course_code để join với weights
    data["course_code_norm"] = data["course_code"].astype(str).str.strip()

    for course, comp_weights in weights_by_course.items():
        mask_course = data["course_code_norm"] == course
        if not mask_course.any():
            continue

        idx = data.index[mask_course]
        baseline = pd.Series(0.0, index=idx)

        for comp, w in comp_weights.items():
            if comp == TARGET_COL:
                # không dùng final làm baseline
                continue
            if comp in data.columns:
                baseline += data.loc[idx, comp].astype(float).fillna(0.0) * w

        data.loc[idx, "baseline_final_weighted"] = baseline

    # Không cần cột norm nữa
    data = data.drop(columns=["course_code_norm"])
    return data

# =======================================
# 3. Hàm xử lý 1 file course
# =======================================

def load_and_process_course_file(path: str) -> pd.DataFrame:
    """
    Đọc một file CSV (1 môn), chuẩn hóa cột, map sang schema chuẩn,
    tạo mask, trả về DataFrame thống nhất.
    """
    print(f"Đang đọc file: {path}")
    df_raw = pd.read_csv(path)

    # Drop cột rác Unnamed
    unnamed_cols = [c for c in df_raw.columns if normalize_col_name(c).startswith("unnamed")]
    if unnamed_cols:
        print(f"  - Drop cột rác: {unnamed_cols}")
        df_raw = df_raw.drop(columns=unnamed_cols)

    # Chuẩn hóa tên cột
    original_cols = list(df_raw.columns)
    col_norm_map = {c: normalize_col_name(c) for c in original_cols}
    df_raw = df_raw.rename(columns=col_norm_map)

    # Map alias thủ công cho student_id, course_code, no nếu cần
    if "student_id" not in df_raw.columns:
        for c in list(df_raw.columns):
            if find_canonical_for_raw(c) == "student_id":
                df_raw = df_raw.rename(columns={c: "student_id"})
                break

    if "course_code" not in df_raw.columns:
        for c in list(df_raw.columns):
            if find_canonical_for_raw(c) == "course_code":
                df_raw = df_raw.rename(columns={c: "course_code"})
                break

    if "no" not in df_raw.columns:
        for c in list(df_raw.columns):
            if find_canonical_for_raw(c) == "no":
                df_raw = df_raw.rename(columns={c: "no"})
                break

    # Xử lý student_id: ép sang string
    if "student_id" in df_raw.columns:
        df_raw["student_id"] = df_raw["student_id"].astype(str)
    else:
        df_raw["student_id"] = df_raw.index.astype(str)

    # Xử lý course_code: nếu không có, lấy tên file làm course_code
    if "course_code" not in df_raw.columns:
        fname = os.path.basename(path)
        base, _ = os.path.splitext(fname)
        df_raw["course_code"] = base

    # Tạo DataFrame output theo schema chuẩn
    df_out = pd.DataFrame(index=df_raw.index)
    df_out["student_id"] = df_raw["student_id"]
    df_out["course_code"] = df_raw["course_code"]
    df_out["no"] = df_raw["no"] if "no" in df_raw.columns else np.arange(len(df_raw)) + 1

    # Ghi nhận cột nào course này thực sự có
    course_has_feature = {f: False for f in CANONICAL_FEATURES}

    # Tạm lưu mapping canonical -> raw_col
    temp_feature_cols: Dict[str, str] = {}

    for raw_col in df_raw.columns:
        canonical = find_canonical_for_raw(raw_col)
        if canonical in CANONICAL_FEATURES:
            temp_feature_cols[canonical] = raw_col
            course_has_feature[canonical] = True

    # Xử lý đặc biệt quiz1/quiz2, homework1/homework2 → tạo quiz/homework chung
    if "quiz1" in df_raw.columns and "quiz2" in df_raw.columns:
        df_out["quiz"] = df_raw["quiz1"].combine(df_raw["quiz2"], np.nanmean)
        course_has_feature["quiz"] = True

    if "homework1" in df_raw.columns and "homework2" in df_raw.columns:
        df_out["homework"] = df_raw["homework1"].combine(df_raw["homework2"], np.nanmean)
        course_has_feature["homework"] = True

    # Copy dữ liệu từ raw -> canonical
    for feat in CANONICAL_FEATURES:
        if feat in df_out.columns:
            # đã được tạo ở trên (quiz/homework gộp)
            continue
        if feat in temp_feature_cols:
            raw_col = temp_feature_cols[feat]
            df_out[feat] = pd.to_numeric(df_raw[raw_col], errors="coerce")
        else:
            df_out[feat] = np.nan  # course này không có cột này

    # Tạo mask cho từng feature (kể cả final, để biết môn có cột final hay không)
    for feat in CANONICAL_FEATURES:
        mask_col = f"{feat}_mask"
        if course_has_feature[feat]:
            df_out[mask_col] = 1
        else:
            df_out[mask_col] = 0

    # Loại bỏ các dòng quá "rác" (rất ít điểm)
    feature_cols_only = CANONICAL_FEATURES.copy()
    non_null_count = df_out[feature_cols_only].notnull().sum(axis=1)
    before = len(df_out)
    df_out = df_out[non_null_count >= 2].copy()
    after = len(df_out)
    if before != after:
        print(f"  - Loại {before - after} dòng có quá ít điểm (non-null < 2)")

    return df_out.reset_index(drop=True)

# =======================================
# 4. Đọc toàn bộ folder & merge
# =======================================

def load_all_courses(data_dir: str) -> pd.DataFrame:
    csv_files = glob.glob(os.path.join(data_dir, "*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"Không tìm thấy CSV nào trong thư mục: {data_dir}")

    all_dfs = []
    for path in csv_files:
        # Bỏ qua file weights nếu nằm chung thư mục
        if os.path.basename(path).startswith("weights_used"):
            continue
        df_course = load_and_process_course_file(path)
        all_dfs.append(df_course)

    data = pd.concat(all_dfs, ignore_index=True)
    print("Kích thước data sau merge:", data.shape)
    return data

# Load dữ liệu courses
data = load_all_courses(DATA_DIR)

# Load weights & tính baseline_final_weighted
weights_by_course = load_weights(WEIGHTS_FILE)
data = add_baseline_from_weights(data, weights_by_course)

# Lưu synthetic_training_sample.csv cho dễ debug/manual check
synthetic_path = os.path.join(OUTPUT_DIR, "synthetic_training_sample_v2.csv")
data.to_csv(synthetic_path, index=False, encoding="utf-8-sig")
print(f"Đã lưu synthetic_training_sample_v2.csv tại: {synthetic_path}")

Đang đọc file: output_final_fixed\CHE 101.csv
Đang đọc file: output_final_fixed\CMU-CS 246.csv
Đang đọc file: output_final_fixed\CMU-CS 252.csv
Đang đọc file: output_final_fixed\CMU-CS 297.csv
Đang đọc file: output_final_fixed\CMU-CS 303.csv
Đang đọc file: output_final_fixed\CMU-CS 311.csv
Đang đọc file: output_final_fixed\CMU-CS 316.csv
Đang đọc file: output_final_fixed\CMU-CS 445.csv
Đang đọc file: output_final_fixed\CMU-CS 447.csv
Đang đọc file: output_final_fixed\CMU-CS 462.csv
Đang đọc file: output_final_fixed\CMU-ENG 130.csv
  - Loại 1 dòng có quá ít điểm (non-null < 2)
Đang đọc file: output_final_fixed\CMU-ENG 230.csv
Đang đọc file: output_final_fixed\CMU-IS 401.csv
Đang đọc file: output_final_fixed\CMU-IS 432.csv
Đang đọc file: output_final_fixed\CMU-SE 100.csv
Đang đọc file: output_final_fixed\CMU-SE 214.csv
Đang đọc file: output_final_fixed\CMU-SE 252.csv
Đang đọc file: output_final_fixed\CMU-SE 303.csv
Đang đọc file: output_final_fixed\COM 141.csv
Đang đọc file: output_final

In [12]:

# =======================================
# 5. Chuẩn bị dữ liệu train/test (KHÔNG leak final)
# =======================================

# Chỉ dùng những dòng có final thật để train
data_train = data[~data[TARGET_COL].isna()].copy()
print("Số dòng có final (dùng để train):", len(data_train))

# Nếu baseline có NaN, fill bằng một giá trị cố định và LƯU lại
if "baseline_final_weighted" not in data_train.columns:
    data_train["baseline_final_weighted"] = np.nan

# Chọn baseline_fill_value: 
# - Cách 1: mean (data_train["baseline_final_weighted"])
# - Cách 2: hẳn một số cố định, ví dụ 5.0
baseline_fill_value = data_train["baseline_final_weighted"].mean()
if np.isnan(baseline_fill_value):
    baseline_fill_value = 5.0  # fallback an toàn

data_train["baseline_final_weighted"] = data_train["baseline_final_weighted"].fillna(baseline_fill_value)


# Tạo danh sách cột feature & mask
feature_cols = FEATURE_SCORE_COLS.copy()   # KHÔNG có 'final'
mask_cols = [f"{f}_mask" for f in CANONICAL_FEATURES]  # mask cho tất cả (kể cả final_mask)

# Numeric features: điểm + mask + baseline
X_num = data_train[feature_cols + mask_cols + ["baseline_final_weighted"]].copy()
X_num = X_num.fillna(0.0)  # phòng hờ NaN

# One-hot cho course_code
course_dummies = pd.get_dummies(data_train["course_code"], prefix="course")
X = pd.concat([X_num, course_dummies], axis=1)

# Target
y = data_train[TARGET_COL].astype(float)

print("Shape X:", X.shape, "| Shape y:", y.shape)

# Lưu lại index để đánh giá theo course_code ở valid set
idx_all = data_train.index.to_numpy()

X_train, X_valid, y_train, y_valid, idx_train, idx_valid = train_test_split(
    X, y, idx_all, test_size=0.2, random_state=42
)

# =======================================
# 6. Train 2 model: RandomForest & GradientBoosting
# =======================================

# ---- Model 1: RandomForest ----
rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_valid)
mae_rf = mean_absolute_error(y_valid, y_pred_rf)
mse_rf = mean_squared_error(y_valid, y_pred_rf)
rmse_rf = np.sqrt(mse_rf)
r2_rf = r2_score(y_valid, y_pred_rf)

print("==== Kết quả RandomForest (KHÔNG leak final) ====")
print(f"MAE  : {mae_rf:.4f}")
print(f"RMSE : {rmse_rf:.4f}")
print(f"R^2  : {r2_rf:.4f}")

# ---- Model 2: GradientBoosting ----
gb_model = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gb_model.fit(X_train, y_train)

y_pred_gb = gb_model.predict(X_valid)
mae_gb = mean_absolute_error(y_valid, y_pred_gb)
mse_gb = mean_squared_error(y_valid, y_pred_gb)
rmse_gb = np.sqrt(mse_gb)
r2_gb = r2_score(y_valid, y_pred_gb)

print("==== Kết quả GradientBoosting (KHÔNG leak final) ====")
print(f"MAE  : {mae_gb:.4f}")
print(f"RMSE : {rmse_gb:.4f}")
print(f"R^2  : {r2_gb:.4f}")

# =======================================
# 7. Bảng metrics theo từng course_code (dựa trên tập VALID)
# =======================================

def eval_by_course(
    data_train: pd.DataFrame,
    y_true_all: pd.Series,
    y_pred_all: np.ndarray,
    idx_used: np.ndarray,
    course_col: str = "course_code"
) -> pd.DataFrame:
    """
    Đánh giá MAE/RMSE/R2 theo từng course_code,
    trên subset được chọn bởi idx_used (vd: idx_valid).
    """
    # Subset theo index valid
    df_sub = data_train.loc[idx_used].copy()
    y_true_sub = y_true_all.loc[idx_used]
    y_pred_sub = pd.Series(y_pred_all, index=idx_used)

    rows = []
    for course, sub in df_sub.groupby(course_col):
        idx = sub.index
        yt = y_true_sub.loc[idx]
        yp = y_pred_sub.loc[idx]
        if len(sub) < 3:
            # quá ít sample, metric sẽ không stable, nhưng vẫn báo
            mae_c = mean_absolute_error(yt, yp)
            mse_c = mean_squared_error(yt, yp)
            rmse_c = np.sqrt(mse_c)
            try:
                r2_c = r2_score(yt, yp)
            except Exception:
                r2_c = np.nan
        else:
            mae_c = mean_absolute_error(yt, yp)
            mse_c = mean_squared_error(yt, yp)
            rmse_c = np.sqrt(mse_c)
            r2_c = r2_score(yt, yp)
        rows.append((course, len(sub), mae_c, rmse_c, r2_c))
    return pd.DataFrame(rows, columns=["course_code", "n_valid", "MAE", "RMSE", "R2"])

print("\n==== Metrics theo course_code - RandomForest (valid set) ====")
df_course_rf = eval_by_course(data_train, y, y_pred_rf, idx_valid)
print(df_course_rf.sort_values("R2"))

print("\n==== Metrics theo course_code - GradientBoosting (valid set) ====")
df_course_gb = eval_by_course(data_train, y, y_pred_gb, idx_valid)
print(df_course_gb.sort_values("R2"))

# Lưu ra file CSV để xem kỹ trong Excel
df_course_rf.to_csv(os.path.join(OUTPUT_DIR, "course_metrics_random_forest.csv"), index=False, encoding="utf-8-sig")
df_course_gb.to_csv(os.path.join(OUTPUT_DIR, "course_metrics_gradient_boosting.csv"), index=False, encoding="utf-8-sig")

# =======================================
# 8. Dự đoán final cho toàn bộ data (dùng model bạn thích, ví dụ RandomForest)
# =======================================

# Chuẩn bị baseline cho toàn bộ data
if "baseline_final_weighted" not in data.columns:
    data["baseline_final_weighted"] = np.nan

baseline_mean_all = data["baseline_final_weighted"].mean()
data["baseline_final_weighted"] = data["baseline_final_weighted"].fillna(baseline_mean_all)

# X_full cho toàn bộ data
X_num_full = data[feature_cols + mask_cols + ["baseline_final_weighted"]].copy()
X_num_full = X_num_full.fillna(0.0)

course_dummies_full = pd.get_dummies(data["course_code"], prefix="course")

# Đồng bộ cột one-hot
for col in course_dummies.columns:
    if col not in course_dummies_full.columns:
        course_dummies_full[col] = 0
course_dummies_full = course_dummies_full[course_dummies.columns]

X_full = pd.concat([X_num_full, course_dummies_full], axis=1)

# Dự đoán bằng RandomForest (hoặc đổi sang gb_model nếu bạn thích)
data["pred_final_rf"] = rf_model.predict(X_full)
data["pred_final_gb"] = gb_model.predict(X_full)

pred_csv_path = os.path.join(OUTPUT_DIR, "predictions_with_final_v2.csv")
data.to_csv(pred_csv_path, index=False, encoding="utf-8-sig")
print(f"\nĐã lưu dự đoán (RF + GB) vào: {pred_csv_path}")
print("\n==== Metrics theo course_code - GradientBoosting (valid set) ====")
df_course_gb = eval_by_course(data_train, y, y_pred_gb, idx_valid)
print(df_course_gb.sort_values("R2"))

Số dòng có final (dùng để train): 4830
Shape X: (4830, 77) | Shape y: (4830,)
==== Kết quả RandomForest (KHÔNG leak final) ====
MAE  : 1.2464
RMSE : 1.7199
R^2  : 0.6052
==== Kết quả GradientBoosting (KHÔNG leak final) ====
MAE  : 1.2707
RMSE : 1.6907
R^2  : 0.6185

==== Metrics theo course_code - RandomForest (valid set) ====
    course_code  n_valid       MAE      RMSE        R2
25       ES 221       12  1.658153  2.205942 -0.526645
41      MTH 341       13  2.120495  2.514674 -0.510382
7    CMU-CS 445       11  1.524972  1.798405 -0.283352
35      MTH 103       20  1.647106  1.955276 -0.126140
22       CS 466       12  1.408210  1.912375 -0.107769
30   IS-ENG 137       16  1.035744  1.299029 -0.102954
43      PHI 150       34  1.617044  2.102131 -0.074259
47      POS 361       39  1.027537  1.371658 -0.072191
46      POS 351       38  2.113544  2.662440 -0.069982
14   CMU-SE 100       10  1.850289  2.566001 -0.052135
20       CS 201       15  2.420181  3.214359 -0.047673
32   IS-ENG

In [13]:
# =======================================
# 8b. Inference với fallback dựa trên course_R2 (GradientBoosting)
# =======================================

# Map course_code -> R2 (GB) từ bảng metrics
course_r2_gb = df_course_gb.set_index("course_code")["R2"].to_dict()

# Đảm bảo baseline đã được fill trước đó
if "baseline_final_weighted" not in data.columns:
    data["baseline_final_weighted"] = np.nan

baseline_mean_all = data["baseline_final_weighted"].mean()
data["baseline_final_weighted"] = data["baseline_final_weighted"].fillna(baseline_mean_all)

# Gắn course_R2_gb vào từng dòng
data["course_R2_gb"] = data["course_code"].map(course_r2_gb)

# Ngưỡng fallback (bạn có thể chỉnh)
HIGH_R2 = 0.60   # tin model
MID_R2 = 0.30    # tin vừa vừa, dùng blend

# Mặc định: nếu không có R2 (course mới), xem như low-confidence
data["course_R2_gb"] = data["course_R2_gb"].fillna(-999)

# Khởi tạo cột final_pred & source & confidence
data["final_pred"] = np.nan
data["pred_source"] = "unknown"
data["confidence_level"] = "unknown"

# Các mask theo R2
mask_high = data["course_R2_gb"] >= HIGH_R2           # tin model
mask_mid  = (data["course_R2_gb"] >= MID_R2) & (data["course_R2_gb"] < HIGH_R2)
mask_low  = data["course_R2_gb"] < MID_R2             # fallback baseline / course mới

# HIGH: dùng thẳng pred_final_gb
data.loc[mask_high, "final_pred"] = data.loc[mask_high, "pred_final_gb"]
data.loc[mask_high, "pred_source"] = "gb_model"
data.loc[mask_high, "confidence_level"] = "high"

# MID: blend giữa model và baseline (70% model, 30% baseline)
alpha = 0.7
blend_mid = (
    alpha * data.loc[mask_mid, "pred_final_gb"] +
    (1 - alpha) * data.loc[mask_mid, "baseline_final_weighted"]
)
data.loc[mask_mid, "final_pred"] = blend_mid
data.loc[mask_mid, "pred_source"] = "gb_baseline_blend"
data.loc[mask_mid, "confidence_level"] = "medium"

# LOW: fallback hoàn toàn về baseline
data.loc[mask_low, "final_pred"] = data.loc[mask_low, "baseline_final_weighted"]
data.loc[mask_low, "pred_source"] = "baseline_only"
data.loc[mask_low, "confidence_level"] = "low"

# ✅ Tới đây, cột final_pred là prediction cuối cùng sau fallback
# Bạn có thể xem nhanh phân bố theo confidence_level:
print("\nSố lượng prediction theo confidence_level:")
print(data["confidence_level"].value_counts())

# Lưu file mới với final_pred
pred_csv_path_fallback = os.path.join(OUTPUT_DIR, "predictions_with_fallback_v2.csv")
data.to_csv(pred_csv_path_fallback, index=False, encoding="utf-8-sig")
print(f"Đã lưu dự đoán với fallback vào: {pred_csv_path_fallback}")



Số lượng prediction theo confidence_level:
confidence_level
low       2307
high      1643
medium     880
Name: count, dtype: int64
Đã lưu dự đoán với fallback vào: ./output_final_v2\predictions_with_fallback_v2.csv


In [14]:
import joblib

artifacts = {
    "rf_model": rf_model,
    "gb_model": gb_model,
    "feature_cols": feature_cols,
    "mask_cols": mask_cols,
    "course_dummy_cols": list(course_dummies.columns),
    "baseline_fill_value": baseline_fill_value,  # ✅ thêm dòng này
}

model_path = os.path.join(OUTPUT_DIR, "model_artifacts_gb_rf.joblib")
joblib.dump(artifacts, model_path)
print(f"Đã lưu model & artifacts vào: {model_path}")


Đã lưu model & artifacts vào: ./output_final_v2\model_artifacts_gb_rf.joblib


In [15]:
model_path_pkl = os.path.join(OUTPUT_DIR, "model_artifacts_gb_rf.pkl")
joblib.dump(artifacts, model_path_pkl)
print(f"Đã lưu model & artifacts (pkl) vào: {model_path_pkl}")


Đã lưu model & artifacts (pkl) vào: ./output_final_v2\model_artifacts_gb_rf.pkl


In [16]:
def build_X_from_df_for_inference(
    df: pd.DataFrame,
    artifacts: dict,
    weights_by_course: dict
) -> (pd.DataFrame, pd.DataFrame):
    """
    df: DataFrame đã chuẩn hóa schema (output của load_and_process_course_file)
    artifacts: dict load từ joblib (chứa feature_cols, mask_cols, course_dummy_cols, baseline_fill_value, ...)
    weights_by_course: weights_used đã load sẵn

    Trả về:
      - df_features: df sau khi thêm baseline_final_weighted (giữ meta + features)
      - X_new: ma trận feature đúng shape cho model.predict(...)
    """
    feature_cols = artifacts["feature_cols"]
    mask_cols = artifacts["mask_cols"]
    course_dummy_cols = artifacts["course_dummy_cols"]
    baseline_fill_value = artifacts.get("baseline_fill_value", 5.0)

    # Thêm baseline_final_weighted cho df (nếu có weight)
    df = add_baseline_from_weights(df, weights_by_course)

    # Nếu course này không có weight → baseline sẽ NaN → fill = baseline_fill_value của TRAIN
    if "baseline_final_weighted" not in df.columns:
        df["baseline_final_weighted"] = np.nan

    df["baseline_final_weighted"] = df["baseline_final_weighted"].fillna(baseline_fill_value)

    # Numeric features: điểm + mask + baseline
    X_num = df[feature_cols + mask_cols + ["baseline_final_weighted"]].copy()
    X_num = X_num.fillna(0.0)

    # One-hot course_code cho file mới
    course_dummies_new = pd.get_dummies(df["course_code"], prefix="course")

    # Đồng bộ cột one-hot với lúc train
    for col in course_dummy_cols:
        if col not in course_dummies_new.columns:
            course_dummies_new[col] = 0
    course_dummies_new = course_dummies_new[course_dummy_cols]

    X_new = pd.concat([X_num, course_dummies_new], axis=1)
    return df, X_new


In [17]:
import joblib
import pandas as pd
import os

# 1. Load artifacts & weights
artifacts = joblib.load(os.path.join(OUTPUT_DIR, "model_artifacts_gb_rf.joblib"))
weights_by_course_inf = load_weights(WEIGHTS_FILE)

# 2. Nhận file upload (ở đây ví dụ là CMU-CS 462.csv)
uploaded_path = os.path.join(DATA_DIR, "CMU-CS 462.csv")  # thay bằng path thực tế

# 3. Chuẩn hóa file
df_course = load_and_process_course_file(uploaded_path)

# 4. Build X_new
df_features, X_new = build_X_from_df_for_inference(
    df_course,
    artifacts=artifacts,
    weights_by_course=weights_by_course_inf
)

# 5. Chọn model (GB làm model chính)
gb_model_loaded = artifacts["gb_model"]
pred_final = gb_model_loaded.predict(X_new)

df_features["pred_final_gb"] = pred_final

# 6. (Tùy bạn) apply thêm fallback theo R² nếu muốn, hoặc dùng luôn pred_final_gb
output_pred_path = os.path.join(OUTPUT_DIR, "pred_for_uploaded_CMU_CS_462_fixed.csv")
df_features.to_csv(output_pred_path, index=False, encoding="utf-8-sig")
print("Đã lưu dự đoán:", output_pred_path)


Đã load weights cho 47 course.
Đang đọc file: output_final_fixed\CMU-CS 462.csv
Đã lưu dự đoán: ./output_final_v2\pred_for_uploaded_CMU_CS_462_fixed.csv


In [ ]:

# =======================================
# 9. Lưu kết quả vào MongoDB (optional)
# =======================================

def save_predictions_to_mongodb(
    df: pd.DataFrame,
    mongo_uri: str = "mongodb://localhost:27017",
    db_name: str = "student_scores",
    collection_name: str = "predictions_v2_fallback"
):
    client = MongoClient(mongo_uri)
    db = client[db_name]
    coll = db[collection_name]

    records = []

    for _, row in df.iterrows():
        record = {
            "student_id": row.get("student_id"),
            "course_code": row.get("course_code"),
            "no": row.get("no"),
            "scores": {feat: row.get(feat) for feat in CANONICAL_FEATURES},
            "masks": {f"{feat}_mask": row.get(f"{feat}_mask") for feat in CANONICAL_FEATURES},
            "baseline_final_weighted": row.get("baseline_final_weighted"),
            "pred_final_rf": row.get("pred_final_rf"),
            "pred_final_gb": row.get("pred_final_gb"),
            # ✅ prediction cuối cùng sau fallback
            "final_pred": row.get("final_pred"),
            "pred_source": row.get("pred_source"),
            "confidence_level": row.get("confidence_level"),
            "created_at": datetime.utcnow()
        }
        if not pd.isna(row.get(TARGET_COL)):
            record["final_true"] = row.get(TARGET_COL)
        records.append(record)

    if records:
        result = coll.insert_many(records)
        print(f"Đã insert {len(result.inserted_ids)} documents vào MongoDB.")
    else:
        print("Không có record nào để insert.")

# Gọi thử (nếu bạn có MongoDB đang chạy)
try:
    save_predictions_to_mongodb(data)
except Exception as e:
    print("Lỗi khi lưu MongoDB (kiểm tra lại URI/connection):", e)
